In [ ]:
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q datasets bitsandbytes evaluate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


In [ ]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 59.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install -q langchain-core langchain-community langchain-huggingface faiss-cpu datasets bm25s[core] PyStemmer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 752.4/752.4 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip uninstall -y numba jax

Found existing installation: numba 0.60.0
Uninstalling numba-0.60.0:
  Successfully uninstalled numba-0.60.0
Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2


In [ ]:
!rm -rf colab_llm_utils
!git clone -b multiaxial https://github.com/ravy101/colab_llm_utils.git

Cloning into 'colab_llm_utils'...
remote: Enumerating objects: 1224, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 1224 (delta 79), reused 93 (delta 43), pack-reused 1092 (from 1)
Receiving objects: 100% (1224/1224), 799.55 KiB | 13.33 MiB/s, done.
Resolving deltas: 100% (768/768), done.


In [ ]:
import colab_llm_utils
from colab_llm_utils import configs

In [ ]:
!rm -rf colab_rag
!git clone -b main https://github.com/ravy101/colab_rag.git

Cloning into 'colab_rag'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 105 (delta 61), reused 74 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 28.16 KiB | 5.63 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [ ]:
from colab_rag import retriever

In [ ]:
from datasets import load_dataset
import pandas as pd
import os

In [ ]:
import torch, transformers, bitsandbytes, numpy
print(torch.__version__)        # 2.3.1+cu124
print(transformers.__version__) # 4.42.4
print(bitsandbytes.__version__) # 0.41.3
print(numpy.__version__)        # 1.26.4
print(torch.cuda.is_available())# True

2.11.0+cu128
5.9.0
0.49.2
2.0.2
True


In [ ]:
model_config = configs.models.qwen3_8b
dataset_config = configs.datasets.arc_challenge
inference_config = configs.inference.multiple_choice

special_tag = ""
short_name = model_config['model_name'].split('/')[-1]

In [ ]:
model_config

{'model_name': 'Qwen/Qwen3-8B',
 'hf_model_func': transformers.models.auto.modeling_auto.AutoModelForCausalLM,
 'bnb_config': BitsAndBytesConfig {
   "_load_in_4bit": true,
   "_load_in_8bit": false,
   "bnb_4bit_compute_dtype": "bfloat16",
   "bnb_4bit_quant_storage": "uint8",
   "bnb_4bit_quant_type": "nf4",
   "bnb_4bit_use_double_quant": true,
   "llm_int8_enable_fp32_cpu_offload": true,
   "llm_int8_has_fp16_weight": false,
   "llm_int8_skip_modules": null,
   "llm_int8_threshold": 6.0,
   "load_in_4bit": true,
   "load_in_8bit": false,
   "quant_method": "bitsandbytes"
 },
 'block_limit': None}

In [ ]:
THINKING = True
special_prompt_note = ""
if THINKING:
  special_tag = "thinking"
  inference_config['max_new_tokens'] = 1000
  inference_config['temperature']=0.6
  inference_config['top_p'] = 0.95
  inference_config['top_k'] = 20
  inference_config['do_sample'] = True
  inference_config['thinking'] = True
  inference_config['terminators'].remove('\n')
  inference_config['terminators'].remove('\n\n')
  special_prompt_note = "Limit your thinking to 600 tokens or less.\n"


In [ ]:
FOLLOW_UP = False
follow_up_prompt = None
if FOLLOW_UP:
  follow_up_prompt = colab_llm_utils.configs.datasets.axis_follow_up_prompt()


In [ ]:
examples = "Question: Which city is known for its famous 'Opera House' and 'Harbour Bridge'?\nChoices: A. Brisbane.\nB. New York.\nC. Sydney.\nD. Paris.\nAnswer: C\n\n"

In [ ]:
RAG = False
if RAG:
  special_tag = "rag"
  DB_DIR = "/content/drive/My Drive/hybrid_wiki_index"
  hrag = retriever.SimpleHybridRetriever("sentence-transformers/all-MiniLM-L6-v2",
                                          bm25s_path="/content/drive/My Drive/wiki_en_full_index/bm25s_index",
                                          faiss_path=None )
  mcqa_examples = ["[Source 4 | RRF Score: 0.0161 | Subject: unknown subject]\nThe word psychosis has two parts. The first part comes from psyche, which means soul in Ancient Greek. The second part is the ending '-osis', which means illness or unnatural condition. So literally, psychosis means unnatural condition of the soul.\nProvide a short answer without explanation.\nQuestion: Which city is known for its famous 'Opera House' and 'Harbour Bridge'?\nChoices: A. Brisbane.\nB. New York.\nC. Sydney.\nD. Paris.\nAnswer: C\n\n"]
  trivia_examples = ["[Source 4 | RRF Score: 0.0161 | Subject: unknown subject]\nThe word psychosis has two parts. The first part comes from psyche, which means soul in Ancient Greek. The second part is the ending '-osis', which means illness or unnatural condition. So literally, psychosis means unnatural condition of the soul.\nProvide a short answer without explanation.\nQuestion: Which city is known for its famous 'Opera House' and 'Harbour Bridge'?\nShort Answer: Sydney.\n\n"]
  if dataset_config['task_type'] == "qa":
    examples = trivia_examples
  elif dataset_config['task_type'] == "multiple_choice":
    examples = mcqa_examples


In [ ]:
dataset_config

{'clean_name': 'ARC-Challenge',
 'dataset_name': 'ARC-Challenge',
 'dataset_location': 'allenai/ai2_arc',
 'options': ['A', 'B', 'C', 'D', 'E'],
 'subset': 'train+test',
 'task_type': 'multiple_choice',
 'dict_ans': False,
 'shuffle': True,
 'doc_to_text': <function colab_llm_utils.configs.datasets.doc_to_text_arc(item)>,
 'doc_to_ans': <function colab_llm_utils.configs.datasets.doc_to_answer_arc(item)>,
 'doc_to_choices': <function colab_llm_utils.configs.datasets.doc_to_choices_arc(item)>}

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

hf_dataset = load_dataset(dataset_config['dataset_location'], dataset_config['dataset_name'], split=dataset_config['subset'], streaming=False)

README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…):   0%|          | 0.00/190k [00:00<?, ?B/s]

ARC-Challenge/test-00000-of-00001.parque(…):   0%|          | 0.00/204k [00:00<?, ?B/s]

ARC-Challenge/validation-00000-of-00001.(…):   0%|          | 0.00/55.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

In [ ]:
test_data = hf_dataset#[dataset_config['subset']]
if dataset_config["shuffle"]:
    test_data = test_data.shuffle(seed=42)
test_data = test_data.select(range(inference_config['n_samples']))
#test_data= test_data.take(inference_config['n_samples'])

In [ ]:
test_data[4]

{'id': 'Mercury_406800',
 'question': 'Bacteria are organisms that reproduce asexually. What would the traits inherited by a newly produced bacterium be like?',
 'choices': {'text': ['different from the traits of the single parent',
   'the same traits as the single parent',
   'different from the traits of two parents',
   'similar traits as two parents'],
  'label': ['A', 'B', 'C', 'D']},
 'answerKey': 'B'}

In [ ]:
torch.cuda.empty_cache()
model = colab_llm_utils.model_tools.LlamaHelper(model_config, inference_config)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in block.
no attention type in

In [ ]:
torch.cuda.empty_cache()
!nvidia-smi

Tue Jun  2 23:48:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P0             26W /   70W |    6015MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
test_data_list = []
for d in test_data:
  test_data_list.append(d)
  #print(doc_to_text_func(d))
  #print(doc_to_answer_func(d))
  last_prompt = dataset_config['doc_to_text'](d)


In [ ]:
import os
if not os.path.exists(f"/content/drive/MyDrive/phase3/Llama/{dataset_config['clean_name']}/"):
  os.makedirs(f"/content/drive/MyDrive/phase3/Llama/{dataset_config['clean_name']}/")

In [ ]:
progress = 24


In [ ]:
def consolidate_context(retrieval_results, threshold = 0.015, examples = None):
    """
    Consolidates RRF results into a single string for LLM prompting.
    """
    context_blocks = []
    instructions = "Supporting sources may or may not be relevenat to the question.\nSupporting sources may refer to other entities if not specified.\nIgnore unrelated sources.\nIf sources are irrelevant, give your best guess.\nDo not mention sources or context, only output the answer or your best guess.\n"

    for i, entry in enumerate(retrieval_results):
        doc = entry['doc']
        score = entry['score']
        title = "unknown subject"

        if score < threshold:
            continue

        try:
            title = doc.metadata['title']
        except:
            pass

        # We include the index and a header to help the LLM cite its sources
        header = f"[Source {i+1} | RRF Score: {score:.4f} | Subject: {title}]"
        block = f"{header}\n{doc.page_content.strip()}"
        context_blocks.append(block)
    prefix = ""
    if examples:
      for e in examples:
        prefix  = prefix + instructions + e

    return prefix + instructions +"\n".join(context_blocks)



In [ ]:
retriever.consolidate_context = consolidate_context

In [ ]:
from langchain_core.documents import Document

In [ ]:
for i in range(progress, int(inference_config['n_samples']/inference_config['samples_per_file'])):
  prompts = []
  responses = []
  blocks = []
  answers = []
  logit_outs = []
  token_outs = []
  meta = []
  for j in range(i*inference_config['samples_per_file'], (i+1)*inference_config['samples_per_file']):
    prompt = dataset_config['doc_to_text'](test_data_list[j])
    if RAG:
      if dataset_config['task_type'] == "multiple_choice":
        rag_result = hrag.mcqa_hybrid_retrieve(test_data_list[j]['question'], dataset_config['doc_to_choices'](test_data_list[j]))
      else:
          rag_result = hrag.hybrid_retrieve(test_data_list[j]['question'])
      context = retriever.consolidate_context(rag_result, threshold = .016, examples = examples)
      prompt =  context + "\n" + prompt
    else:
      prompt = examples + special_prompt_note + prompt
    #logits, attn, mlp, block = model.logits_and_intermediates(prompt)
    if len(prompt) > 28000:
      print("Cropping *************************************")
      prompt = prompt[:28000]
    print(prompt)
    #print(f"prompt len {len(prompt.split())}")
    output, sequence, logit_dict, deep_probs, meta_info = model.get_sequence_logits(prompt, use_beams=False, follow_up_prompt=follow_up_prompt )
    print(f"Output: {output[0]}")
    blocks.append(deep_probs)
    token_outs.append(sequence)
    prompts.append(prompt)
    answers.append(dataset_config['doc_to_ans'](test_data_list[j]))
    model.reset_all()
    torch.cuda.empty_cache()
    responses.append(output)
    logit_outs.append(logit_dict)
    meta.append(meta_info)
    print(output)
    print(f"expected answer: {dataset_config['doc_to_ans'](test_data_list[j])}")
    print("\n**************************")
  df = pd.DataFrame({"prompts": prompts, "responses": responses, "ans": answers, "logit_outs": logit_outs, "token_outs": token_outs, "blocks": blocks, "meta": meta})
  df.to_pickle(f"/content/drive/MyDrive/phase3/Llama/{dataset_config['clean_name']}/{dataset_config['dataset_name']}_{dataset_config['subset']}_{short_name}{special_tag}_{int(i):04d}.pickle")
  print("saving file.")
print("All done.")

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Question: Which city is known for its famous 'Opera House' and 'Harbour Bridge'?
Choices: A. Brisbane.
B. New York.
C. Sydney.
D. Paris.
Answer: C

Limit your thinking to 600 tokens or less.
Answer the following multiple choice question with only the letter corresponding to the correct answer.
Question: A student investigated the percentage of energy obtained from several food sources for a population of eagles. Which format is the best way to display this data?
Choices: A. a table
B. a pie chart
C. a bar graph
D. a line graph

Answer: 


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Output: <think>
Okay, let's see. The question is about displaying data on the percentage of energy obtained from different food sources by eagles.

First, I need to recall what each type of graph/table is used for. 

A table (option A) lists data in rows and columns. It's good for precise numbers but not so great for showing trends or comparisons at a glance. But since percentages are involved here, maybe they want it organized clearly?

Pie charts (B) show proportions as parts of a whole. Since the question mentions "percentage," that might be relevant because pie charts visually represent how much each category contributes to the total. If the eagle's diet has various food sources, like fish, mammals, etc., a pie chart could show the proportion each takes up.

Bar graphs (C) compare quantities across categories. Each bar represents one food source, with height indicating the percentage. That would work too if comparing which foods contribute more. However, when dealing with percentag

In [ ]:
from google.colab import runtime
runtime.unassign()